## Get historical data

In [2]:
import pandas as pd
import requests
from time import sleep
from tqdm import tqdm

In [3]:
data = pd.read_csv("../data/streaming/bitola_sensor_weather_features_online.csv")

In [4]:
data

,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
0,2025-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.9505,92.41695,4.293669,303.02386,943.69060,34.750000,17.500000
1,2025-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,2.0005,92.41994,3.818376,315.00010,944.07420,18.250000,10.000000
2,2025-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.6505,93.06822,3.893995,326.30990,944.16860,15.000000,8.750000
3,2025-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.4005,93.72927,4.104631,322.12494,944.47410,14.750000,7.250000
4,2025-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127,1.3005,94.06355,3.563818,315.00010,944.81850,19.000000,8.750000
...,...,...,...,...,...,...,...,...,...,...,...
48021,2026-03-01 18:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,6.9890,69.99575,1.659518,229.39879,949.81270,12.250000,4.000000
48022,2026-03-01 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,6.4890,71.15741,1.049571,210.96368,950.14496,5.666667,2.666667
48023,2026-03-01 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,5.6390,72.80048,1.527351,224.99990,950.19970,6.000000,3.000000
48024,2026-03-01 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767,5.1390,72.96915,1.152562,231.34016,950.34550,3.750000,2.250000


In [5]:
df = data[['timestamp','sensorId','lat','lon']]
df

,timestamp,sensorId,lat,lon
0,2025-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
1,2025-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
2,2025-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
3,2025-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
4,2025-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
...,...,...,...,...
48021,2026-03-01 18:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767
48022,2026-03-01 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767
48023,2026-03-01 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767
48024,2026-03-01 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767


In [6]:
df["timestamp"] = pd.to_datetime(df["timestamp"],utc=True)
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour

/tmp/ipykernel_35822/2002426275.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["timestamp"] = pd.to_datetime(df["timestamp"],utc=True)
/tmp/ipykernel_35822/2002426275.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["date"] = df["timestamp"].dt.date
/tmp/ipykernel_35822/2002426275.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.o

In [7]:
def get_data_for_day(lat, lon, date):
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": str(date),
        "end_date": str(date),
        "hourly": ["temperature_2m", "relative_humidity_2m", "surface_pressure", "wind_speed_10m"],
        "timezone": "UTC"
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [8]:
weather_cache = {}

In [9]:
max(df['timestamp'])

Timestamp('2026-03-01 22:00:00+0000', tz='UTC')

In [10]:
df = df[['sensorId', 'lat', 'lon']].drop_duplicates()

# Create hourly timestamps
timestamps = pd.date_range(
    start='2025-11-09 16:00:00',
    end='2026-03-01 22:00:00',
    freq='h'
)

# Cartesian product between sensors and timestamps
new_df = (
    df.assign(key=1)
    .merge(
        pd.DataFrame({'timestamp': timestamps, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)

# Arrange columns
new_df = new_df[['timestamp', 'sensorId', 'lat', 'lon']]

# Combine with your existing dataframe
df = pd.concat([new_df, df], ignore_index=True)

# Sort
df = df.sort_values(['sensorId', 'timestamp']).reset_index(drop=True)

In [11]:
df

,timestamp,sensorId,lat,lon
0,2025-11-09 16:00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
1,2025-11-09 17:00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
2,2025-11-09 18:00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
3,2025-11-09 19:00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
4,2025-11-09 20:00:00,16836a55-7140-43e2-9a63-56fac5cba714,41.023537,21.330127
...,...,...,...,...
59307,2026-03-01 19:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767
59308,2026-03-01 20:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767
59309,2026-03-01 21:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767
59310,2026-03-01 22:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,41.024649,21.320767


In [12]:
df = df[df["timestamp"].notna()]

In [13]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour

/tmp/ipykernel_35822/1979510837.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["date"] = df["timestamp"].dt.date
/tmp/ipykernel_35822/1979510837.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["hour"] = df["timestamp"].dt.hour


In [14]:
wind_values = []
humidity_values = []
pressure_values = []
temperature_values = []

print("\nFetching data from Open-Meteo...\n")

for i, row in tqdm(df.iterrows(), total=len(df)):
    key = (row.lat, row.lon, row.date)

    print(f"Iteration: {i}/{len(df)}")
    if key not in weather_cache:
        success = False
        for _ in range(3):
            try:
                weather_cache[key] = get_data_for_day(row.lat, row.lon, row.date)
                success = True
                sleep(0.2)
                break
            except Exception as e:
                sleep(2)

        if not success:
            weather_cache[key] = None

    data = weather_cache[key]

    if data is None:
        wind_values.append(None)
        humidity_values.append(None)
        pressure_values.append(None)
        temperature_values.append(None)
        continue

    try:
        times = data["hourly"]["time"]
        winds = data["hourly"]["wind_speed_10m"]
        temperatures = data['hourly']['temperature_2m']
        humidity = data['hourly']['relative_humidity_2m']
        pressure = data['hourly']['surface_pressure']

        # Match exact timestamp hour
        ts_str = row.timestamp.strftime("%Y-%m-%dT%H:00")

        if ts_str in times:
            idx = times.index(ts_str)
            wind_values.append(winds[idx])
            humidity_values.append(humidity[idx])
            pressure_values.append(pressure[idx])
            temperature_values.append(temperatures[idx])
        else:
            wind_values.append(None)
            humidity_values.append(None)
            pressure_values.append(None)
            temperature_values.append(None)

    except:
        wind_values.append(None)
        humidity_values.append(None)
        pressure_values.append(None)
        temperature_values.append(None)


Fetching data from Open-Meteo...



  0%|          | 0/59290 [00:00<?, ?it/s]

Iteration: 0/59290


  0%|          | 1/59290 [00:00<7:24:28,  2.22it/s]

Iteration: 1/59290
Iteration: 2/59290
Iteration: 3/59290
Iteration: 4/59290
Iteration: 5/59290
Iteration: 6/59290
Iteration: 7/59290
Iteration: 8/59290


...

Iteration: 59241/59290
Iteration: 59242/59290
Iteration: 59243/59290
Iteration: 59244/59290
Iteration: 59245/59290
Iteration: 59246/59290
Iteration: 59247/59290
Iteration: 59248/59290
Iteration: 59249/59290
Iteration: 59250/59290
Iteration: 59251/59290
Iteration: 59252/59290
Iteration: 59253/59290
Iteration: 59254/59290
Iteration: 59255/59290
Iteration: 59256/59290
Iteration: 59257/59290
Iteration: 59258/59290
Iteration: 59259/59290
Iteration: 59260/59290
Iteration: 59261/59290
Iteration: 59262/59290
Iteration: 59263/59290
Iteration: 59264/59290


100%|█████████▉| 59244/59290 [45:14<00:01, 45.52it/s]

Iteration: 59265/59290
Iteration: 59266/59290
Iteration: 59267/59290
Iteration: 59268/59290
Iteration: 59269/59290
Iteration: 59270/59290
Iteration: 59271/59290
Iteration: 59272/59290
Iteration: 59273/59290
Iteration: 59274/59290
Iteration: 59275/59290
Iteration: 59276/59290
Iteration: 59277/59290
Iteration: 59278/59290
Iteration: 59279/59290
Iteration: 59280/59290
Iteration: 59281/59290
Iteration: 59282/59290
Iteration: 59283/59290
Iteration: 59284/59290
Iteration: 59285/59290
Iteration: 59286/59290
Iteration: 59287/59290
Iteration: 59288/59290


100%|██████████| 59290/59290 [45:15<00:00, 21.84it/s]

Iteration: 59289/59290
Iteration: 59290/59290
Iteration: 59291/59290
Iteration: 59292/59290
Iteration: 59293/59290
Iteration: 59294/59290
Iteration: 59295/59290
Iteration: 59296/59290
Iteration: 59297/59290
Iteration: 59298/59290
Iteration: 59299/59290
Iteration: 59300/59290
Iteration: 59301/59290
Iteration: 59302/59290
Iteration: 59303/59290
Iteration: 59304/59290
Iteration: 59305/59290
Iteration: 59306/59290
Iteration: 59307/59290
Iteration: 59308/59290
Iteration: 59309/59290
Iteration: 59310/59290


In [15]:
df['humidity'] = humidity_values
df['pressure'] = pressure_values
df['temperature'] = temperature_values
df['wind_speed'] = wind_values
df = df.drop(columns=["date", "hour"])

/tmp/ipykernel_35822/886961149.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['humidity'] = humidity_values
/tmp/ipykernel_35822/886961149.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pressure'] = pressure_values
/tmp/ipykernel_35822/886961149.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/

In [16]:
df = df.drop(columns=["lat", "lon"])

In [17]:
df

,timestamp,sensorId,humidity,pressure,temperature,wind_speed
0,2025-11-09 16:00:00,16836a55-7140-43e2-9a63-56fac5cba714,88,944.7,12.5,7.1
1,2025-11-09 17:00:00,16836a55-7140-43e2-9a63-56fac5cba714,90,944.7,12.1,7.1
2,2025-11-09 18:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.5,12.0,6.2
3,2025-11-09 19:00:00,16836a55-7140-43e2-9a63-56fac5cba714,91,944.9,12.0,5.8
4,2025-11-09 20:00:00,16836a55-7140-43e2-9a63-56fac5cba714,93,944.8,11.8,5.8
...,...,...,...,...,...,...
59306,2026-03-01 18:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,51,950.9,7.6,7.4
59307,2026-03-01 19:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,950.7,6.5,6.2
59308,2026-03-01 20:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,56,951.1,5.8,6.8
59309,2026-03-01 21:00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,58,951.2,5.2,6.9


In [18]:
df.isnull().sum()

timestamp      0
sensorId       0
humidity       0
pressure       0
temperature    0
wind_speed     0
dtype: int64

In [19]:
df.to_csv('../data/raw/bitola_forecast_weather.csv',index=False)